# Food-Sure: Concentration Test — can the models tell *how much* adulterant?

**Why this notebook exists.** The new pipeline (`Untitled5.ipynb`) is a 3-class classifier (pure / adulterant A /
adulterant B) followed by a regressor that estimates the adulteration percentage. The argument for it is that a
regressor can report in-between values (8%, 12%) while the 7-class ResNet50 can only say 5, 10 or 15.
This notebook tests that argument with the photos we already have.

**Three questions**

1. **Same test photos, both models** — on `dataset_split/<spice>/test`, how far off is each model's percentage?
2. **Hide-the-10% test** — train the regressor on pure, 5% and 15% photos only, then show it the 10% photos it has
   never seen. Does it place them *between* 5% and 15%?
3. **Would the same idea work on top of ResNet50?** — same tests using frozen ResNet50 image features.

**How to read the hide-the-10% test.** A regressor that has learned nothing answers "about 10" for every adulterated
photo, so "it said 10 for the 10% photos" proves nothing by itself. What matters is whether it gives 5% photos a
lower number than 10% photos, and 10% photos a lower number than 15% photos. That is the *ordering score* below:
50 = coin flip, 100 = always in the right order.

**Read-only with respect to existing work.** Uses the cached features in `Food Sure/fair_comparison/` and the
deployed models in `web_models/`. Outputs go to a new folder `Food Sure/fair_comparison/concentration_test/`.

## 1 — Setup

In [ ]:
import os, sys, subprocess
LOCAL = os.environ.get('FS_LOCAL') == '1'      # only used for an offline dry run with dummy data
if not LOCAL:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'catboost', 'lightgbm'], check=True)

In [ ]:
import re, json, warnings
import numpy as np, pandas as pd
from scipy.stats import skew, kurtosis, spearmanr
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, VotingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score
import catboost as cb
import lightgbm as lgb
import matplotlib
if LOCAL:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR       = os.environ.get('FS_BASE', '/content/drive/MyDrive/Food Sure')
SPLIT_DIR      = os.path.join(BASE_DIR, 'dataset_split')
WEB_MODELS_DIR = os.path.join(BASE_DIR, 'web_models')
FEAT_DIR       = os.path.join(BASE_DIR, 'fair_comparison')                 # cached handcrafted features live here
OUT_DIR        = os.path.join(FEAT_DIR, 'concentration_test')              # new folder, only outputs go here
os.makedirs(OUT_DIR, exist_ok=True)

SPICES = ['black_pepper_powder', 'red_chilli_powder', 'turmeric']
SEED = 42
CROP_SIZE, CROPS_PER_IMAGE = 256, 4
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')
HIDDEN_LEVEL = 10.0

for s in SPICES:
    for part in ['train', 'val', 'test']:
        assert os.path.isdir(os.path.join(SPLIT_DIR, s, part)), f'missing {s}/{part}'
print('Paths OK')

## 2 — Data helpers and the new pipeline's code
Feature extractor, crops, classifier and regressor settings are copied unchanged from `Untitled5.ipynb`.

In [ ]:
def to_type(label):
    label = str(label).lower()
    return 'pure' if label == 'pure' else re.sub(r'^\d+_pct_', '', label)

def pct_of(label):
    m = re.match(r'^(\d+)_pct_', str(label).lower())
    return float(m.group(1)) if m else 0.0

def list_split(spice, part):
    root = os.path.join(SPLIT_DIR, spice, part)
    items = []
    for cls in sorted(os.listdir(root)):
        d = os.path.join(root, cls)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(VALID_EXTS):
                items.append((os.path.join(d, f), cls.lower()))
    return items


def extract_80plus_features(img_bgr, use_gray_world=False):
    import cv2
    from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
    if use_gray_world:
        b, g, r = cv2.split(img_bgr.astype(np.float32))
        gray_avg = (np.mean(b) + np.mean(g) + np.mean(r)) / 3.0
        b = np.clip(b * (gray_avg / (np.mean(b) + 1e-5)), 0, 255)
        g = np.clip(g * (gray_avg / (np.mean(g) + 1e-5)), 0, 255)
        r = np.clip(r * (gray_avg / (np.mean(r) + 1e-5)), 0, 255)
        img_bgr = cv2.merge([b, g, r]).astype(np.uint8)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    img_ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    feats = []
    for space in [img_rgb, img_hsv, img_lab, img_ycrcb]:
        for ch in range(3):
            plane = space[:, :, ch].astype(np.float32).ravel()
            mean_v, std_v = np.mean(plane), np.std(plane)
            sk_v = float(skew(plane)) if std_v > 1e-3 else 0.0
            kt_v = float(kurtosis(plane)) if std_v > 1e-3 else 0.0
            feats.extend([mean_v, std_v, sk_v, kt_v])
    h_hist, _ = np.histogram(img_hsv[:, :, 0], bins=16, range=(0, 180), density=True)
    s_hist, _ = np.histogram(img_hsv[:, :, 1], bins=16, range=(0, 256), density=True)
    l_hist, _ = np.histogram(img_lab[:, :, 0], bins=16, range=(0, 256), density=True)
    a_hist, _ = np.histogram(img_lab[:, :, 1], bins=16, range=(0, 256), density=True)
    b_hist, _ = np.histogram(img_lab[:, :, 2], bins=16, range=(0, 256), density=True)
    feats.extend(np.hstack([h_hist, s_hist, l_hist, a_hist, b_hist]))
    lab_l = img_lab[:, :, 0].astype(np.float32)
    lab_a = img_lab[:, :, 1].astype(np.float32)
    lab_b = img_lab[:, :, 2].astype(np.float32)
    hsv_s = img_hsv[:, :, 1].astype(np.float32)
    cai = np.mean(lab_b) / (np.mean(lab_l) + 1e-5)
    rpi = np.mean(lab_a) / (np.mean(hsv_s) + 1e-5)
    fleck_ratio = np.sum(img_gray > 140) / (img_gray.size + 1e-5)
    lap_var = float(np.var(cv2.Laplacian(img_gray, cv2.CV_64F)))
    feats.extend([cai, rpi, fleck_ratio, lap_var])
    glcm = graycomatrix(img_gray, distances=[1, 2], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    glcm_iso = np.mean(glcm, axis=3, keepdims=True)
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        feats.extend(graycoprops(glcm_iso, prop).ravel())
    lbp = local_binary_pattern(img_gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
    feats.extend(lbp_hist)
    return np.array(feats, dtype=np.float32)


def image_crops(img):
    import cv2
    h, w = img.shape[:2]
    if h >= CROP_SIZE and w >= CROP_SIZE:
        coords = [((h - CROP_SIZE) // 2, (w - CROP_SIZE) // 2), (0, 0),
                  (h - CROP_SIZE, w - CROP_SIZE), (0, w - CROP_SIZE)]
        return [img[t:t + CROP_SIZE, l:l + CROP_SIZE] for t, l in coords[:CROPS_PER_IMAGE]]
    return [cv2.resize(img, (CROP_SIZE, CROP_SIZE))]


def featurize(spice, part_name, items, use_gray_world):
    # Features per crop; groups[i] = index of the source photo in `items`. Cached (same files as the fair comparison).
    cache = os.path.join(FEAT_DIR, f'feats_{spice}_{part_name}_gw{int(use_gray_world)}.npz')
    if os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        return d['X'], d['y'], d['groups']
    import cv2
    from tqdm import tqdm
    X, y, groups = [], [], []
    for gid, (p, label) in enumerate(tqdm(items, desc=f'{spice}/{part_name}')):
        img = cv2.imread(p)
        if img is None:
            print('unreadable, skipped:', p)
            continue
        for patch in image_crops(img):
            X.append(extract_80plus_features(patch, use_gray_world=use_gray_world))
            y.append(label)
            groups.append(gid)
    X, y, groups = np.array(X), np.array(y), np.array(groups)
    np.savez(cache, X=X, y=y, groups=groups)
    return X, y, groups


def load_spice(spice):
    # All photos of one spice. Photo id = position in train+val+test listing; ids >= n_tv are the test photos.
    gw = (spice == 'red_chilli_powder')
    tv, te = list_split(spice, 'train') + list_split(spice, 'val'), list_split(spice, 'test')
    Xa, ya, ga = featurize(spice, 'trainval', tv, gw)
    Xb, yb, gb = featurize(spice, 'test', te, gw)
    y7 = np.concatenate([ya, yb]).astype(str)
    return {'spice': spice, 'items': tv + te, 'n_tv': len(tv),
            'X': np.vstack([Xa, Xb]).astype(np.float32),
            'g': np.concatenate([ga, gb + len(tv)]).astype(int),
            'y3': np.array([to_type(l) for l in y7]),
            'pct': np.array([pct_of(l) for l in y7])}


def make_clf(spice):
    if spice == 'red_chilli_powder':  # Pipeline 2
        m1 = cb.CatBoostClassifier(iterations=550, learning_rate=0.035, depth=5, auto_class_weights='Balanced', verbose=0, random_seed=42)
        m2 = lgb.LGBMClassifier(n_estimators=350, learning_rate=0.035, max_depth=5, class_weight='balanced', random_state=42, verbose=-1)
        m3 = ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42)
        return VotingClassifier(estimators=[('cb', m1), ('lgb', m2), ('et', m3)], voting='soft')
    return cb.CatBoostClassifier(iterations=600, learning_rate=0.04, depth=5, auto_class_weights='Balanced', verbose=0, random_seed=42)


def classifier_fit_predict(spice, X_tr, y_tr, X_te, groups_te):
    # Stage 1 of the new pipeline. Returns (photo ids, class names, per-photo probabilities averaged over crops)
    scaler = RobustScaler().fit(X_tr)
    X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)
    selector = SelectKBest(score_func=f_classif, k=min(65, X_tr.shape[1])).fit(X_tr_s, y_tr)
    clf = make_clf(spice)
    clf.fit(selector.transform(X_tr_s), y_tr)
    probs = clf.predict_proba(selector.transform(X_te_s))
    ids = np.unique(groups_te)
    return ids, np.array(clf.classes_).astype(str), np.stack([probs[groups_te == g].mean(axis=0) for g in ids])


def fit_extratrees(X_tr, pct_tr):
    # Stage 2 of the new pipeline, settings unchanged: RobustScaler -> SelectKBest(f_regression, 45) -> ExtraTrees
    scaler = RobustScaler().fit(X_tr)
    Xs = scaler.transform(X_tr)
    sel = SelectKBest(score_func=f_regression, k=min(45, X_tr.shape[1])).fit(Xs, pct_tr)
    reg = ExtraTreesRegressor(n_estimators=300, max_depth=9, min_samples_split=4, min_samples_leaf=2,
                              random_state=42, n_jobs=-1).fit(sel.transform(Xs), pct_tr)
    return lambda X: reg.predict(sel.transform(scaler.transform(X)))


def fit_ridge(X_tr, pct_tr):
    # A plain linear regressor, as a second opinion: straight lines can interpolate, trees can only average
    m = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 13))).fit(X_tr, pct_tr)
    return m.predict

REGRESSORS = {'ExtraTrees (new pipeline)': fit_extratrees, 'Ridge (linear)': fit_ridge}

def photo_median(values, g, ids):
    return np.array([np.median(values[g == i]) for i in ids])

## 3 — ResNet50: the deployed models, and frozen image features
* **Deployed ResNet50** (`web_models/`): its 7 class probabilities give a percentage two ways — the top class
  (what the app shows today) and a probability-weighted average of 5/10/15.
* **Frozen ResNet50 features**: the standard ImageNet ResNet50 with no fine-tuning, used only as a feature
  extractor (2048 numbers per photo). It has never seen our labels, so it can be used in the hide-the-10% test.
  The deployed models cannot: they were trained with the 10% photos.

In [ ]:
def resnet_predict(spice, items):
    with open(os.path.join(WEB_MODELS_DIR, spice, 'label_map.json')) as f:
        label_map = json.load(f)
    classes = np.array([label_map[str(i)].lower() for i in range(len(label_map))])
    if LOCAL:   # dry run: noisy but informative fake probabilities
        rng = np.random.default_rng(1)
        probs = rng.dirichlet(np.ones(len(classes)), len(items))
        for k, (_, lab) in enumerate(items):
            probs[k, list(classes).index(lab)] += 1.0
        return classes, probs / probs.sum(1, keepdims=True)
    import tensorflow as tf
    model = tf.keras.models.load_model(os.path.join(WEB_MODELS_DIR, spice, 'model.keras'))
    batch = np.stack([tf.keras.utils.img_to_array(tf.keras.utils.load_img(p, target_size=(224, 224)))
                      for p, _ in items])
    probs = model.predict(batch, batch_size=16, verbose=0)
    del model
    tf.keras.backend.clear_session()
    return classes, probs


def collapse_to_type(classes, probs):
    types = np.array([to_type(c) for c in classes])
    tclasses = np.unique(types)
    return tclasses, np.stack([probs[:, types == t].sum(axis=1) for t in tclasses], axis=1)


_backbone = None
def embed_spice(spice, items):
    # One 2048-number vector per photo from the frozen ImageNet ResNet50. Cached to OUT_DIR. NaN row = unreadable.
    global _backbone
    cache = os.path.join(OUT_DIR, f'emb_{spice}.npz')
    if os.path.exists(cache):
        return np.load(cache)['E']
    if LOCAL:
        rng = np.random.default_rng(2)
        E = rng.normal(size=(len(items), 64)).astype(np.float32)
        E[:, 0] += np.array([pct_of(l) for _, l in items]) / 5.0
        return E
    import tensorflow as tf
    from tqdm import tqdm
    if _backbone is None:
        _backbone = tf.keras.applications.ResNet50(include_top=False, weights='imagenet', pooling='avg')
    E = np.full((len(items), 2048), np.nan, dtype=np.float32)
    for start in tqdm(range(0, len(items), 32), desc=f'embedding {spice}'):
        batch, idx = [], []
        for i in range(start, min(start + 32, len(items))):
            try:
                batch.append(tf.keras.utils.img_to_array(tf.keras.utils.load_img(items[i][0], target_size=(224, 224))))
                idx.append(i)
            except Exception as e:
                print('unreadable, skipped:', items[i][0], e)
        if batch:
            x = tf.keras.applications.resnet50.preprocess_input(np.stack(batch))
            E[idx] = _backbone.predict(x, verbose=0)
    np.savez(cache, E=E)
    return E

## 4 — Scoring helpers

In [ ]:
LEVELS = [0.0, 5.0, 10.0, 15.0]

def conc_metrics(true_pct, pred):
    true_pct, pred = np.asarray(true_pct, float), np.asarray(pred, float)
    ad = true_pct > 0
    err = pred - true_pct
    out = {'n_photos': int(len(true_pct)), 'n_adulterated': int(ad.sum()),
           'avg_error_all': float(np.abs(err).mean()),
           'avg_error_adulterated': float(np.abs(err[ad]).mean()),
           'rmse_adulterated': float(np.sqrt((err[ad] ** 2).mean())),
           'right_band_pct': float((np.abs(err[ad]) <= 2.5).mean() * 100)}   # within 2.5 points = nearest of 5/10/15
    rho = spearmanr(true_pct[ad], pred[ad]).correlation if np.std(pred[ad]) > 1e-9 else np.nan
    out['rank_corr_adulterated'] = float(rho)
    for lv in LEVELS:
        m = true_pct == lv
        out[f'avg_answer_at_{int(lv)}'] = float(pred[m].mean()) if m.any() else np.nan
    return out

def boot_mean_ci(values, n=10000, seed=SEED):
    values = np.asarray(values, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), (n, len(values)))
    return [float(v) for v in np.percentile(values[idx].mean(axis=1), [2.5, 97.5])]

def ordering_score(pred, true_pct, low, high):
    # Chance that a photo at `high`% gets a larger answer than a photo at `low`%. 50 = coin flip, 100 = always.
    m = (true_pct == low) | (true_pct == high)
    if (true_pct[m] == low).sum() == 0 or (true_pct[m] == high).sum() == 0:
        return np.nan
    return float(roc_auc_score(true_pct[m] == high, pred[m]) * 100)

def save_summary(summary):
    with open(os.path.join(OUT_DIR, 'summary.txt'), 'w') as f:
        json.dump(summary, f, indent=1, default=lambda o: None if (isinstance(o, float) and np.isnan(o)) else float(o))

summary = {}

## 5 — Question 1: same test photos, how far off is each model's percentage?
Both models are trained on the train+val photos and scored on the same `dataset_split/test` photos.
* **New pipeline, as built**: classifier decides pure / adulterant; if pure the answer is 0, otherwise the
  regressor's answer (clipped to 0.5–16), exactly as in `FoodSureInferenceEngine`.
* **ResNet50, top class**: the percentage of its most likely class (what the app shows today).
* **ResNet50, weighted**: 0 if it says pure, otherwise the probability-weighted average of 5/10/15.
* **Always 10%**: a yardstick that knows nothing — it answers 10 for every adulterated photo.

`avg_error_adulterated` is the average distance from the true percentage on adulterated photos (lower is better).
`right_band_pct` is how often the answer is within 2.5 points of the truth.

In [ ]:
q1_rows, q1_type_rows, q1_pairs, q1_photos = [], [], [], []
data, emb = {}, {}

for spice in SPICES:
    D = load_spice(spice)
    data[spice] = D
    te_rows = D['g'] >= D['n_tv']
    tr_rows = ~te_rows
    te_ids = np.unique(D['g'][te_rows])
    true7 = np.array([D['items'][i][1] for i in te_ids])
    true3 = np.array([to_type(l) for l in true7])
    true_pct = np.array([pct_of(l) for l in true7])
    ad = true_pct > 0

    # New pipeline: stage 1 classifier + stage 2 regressor
    ids, cls3, p3 = classifier_fit_predict(spice, D['X'][tr_rows], D['y3'][tr_rows], D['X'][te_rows], D['g'][te_rows])
    assert np.array_equal(ids, te_ids)
    pipe_type = cls3[p3.argmax(1)]
    reg = fit_extratrees(D['X'][tr_rows], D['pct'][tr_rows])
    reg_raw = photo_median(reg(D['X'][te_rows]), D['g'][te_rows], te_ids)
    pipe_pct = np.where(pipe_type == 'pure', 0.0, np.clip(reg_raw, 0.5, 16.0))

    # Deployed ResNet50 on the same photos
    r_cls7, r_p7 = resnet_predict(spice, D['items'][D['n_tv']:])
    r_p7 = r_p7[te_ids - D['n_tv']]
    cls_pct = np.array([pct_of(c) for c in r_cls7])
    r_cls3, r_p3 = collapse_to_type(r_cls7, r_p7)
    res_type = r_cls3[r_p3.argmax(1)]
    res_top = cls_pct[r_p7.argmax(1)]
    nonpure = cls_pct > 0
    res_weighted = np.where(res_type == 'pure', 0.0,
                            (r_p7[:, nonpure] @ cls_pct[nonpure]) / np.maximum(r_p7[:, nonpure].sum(1), 1e-9))

    answers = {'New pipeline (as built)': pipe_pct,
               'New pipeline regressor alone': reg_raw,
               'ResNet50 top class (app today)': res_top,
               'ResNet50 weighted': res_weighted,
               'Always 10% (yardstick)': np.where(ad, 10.0, 0.0)}
    for name, pred in answers.items():
        lo, hi = boot_mean_ci(np.abs(pred - true_pct)[ad])
        q1_rows.append({'spice': spice, 'model': name, **conc_metrics(true_pct, pred),
                        'avg_error_adult_ci_low': lo, 'avg_error_adult_ci_high': hi})
    for name, t in [('New pipeline', pipe_type), ('ResNet50', res_type)]:
        q1_type_rows.append({'spice': spice, 'model': name, 'n_photos': len(true3),
                             'acc_3class': accuracy_score(true3, t) * 100,
                             'pure_recall': recall_score(true3, t, labels=['pure'], average='macro', zero_division=0) * 100,
                             'adulterated_called_pure': int(np.sum((true3 != 'pure') & (t == 'pure'))),
                             'pure_called_adulterated': int(np.sum((true3 == 'pure') & (t != 'pure')))})
    for other in ['ResNet50 top class (app today)', 'ResNet50 weighted', 'Always 10% (yardstick)']:
        diff = (np.abs(pipe_pct - true_pct) - np.abs(answers[other] - true_pct))[ad]
        lo, hi = boot_mean_ci(diff)
        q1_pairs.append({'spice': spice, 'compared_with': other, 'pipeline_minus_other_avg_error': float(diff.mean()),
                         'ci_low': lo, 'ci_high': hi,
                         'verdict': 'pipeline better' if hi < 0 else ('pipeline worse' if lo > 0 else 'no clear difference')})
    q1_photos.append(pd.DataFrame({'spice': spice, 'photo': [os.path.basename(D['items'][i][0]) for i in te_ids],
                                   'true_label': true7, 'true_pct': true_pct, 'pipeline_type': pipe_type,
                                   'resnet_type': res_type, **{k: np.round(v, 2) for k, v in answers.items()}}))
    print(f'{spice}: done ({len(te_ids)} test photos)')

q1 = pd.DataFrame(q1_rows).round(2)
q1_type = pd.DataFrame(q1_type_rows).round(2)
q1_pairs = pd.DataFrame(q1_pairs).round(2)
q1.to_csv(os.path.join(OUT_DIR, 'q1_concentration_error.csv'), index=False)
q1_type.to_csv(os.path.join(OUT_DIR, 'q1_type_accuracy.csv'), index=False)
q1_pairs.to_csv(os.path.join(OUT_DIR, 'q1_paired.csv'), index=False)
pd.concat(q1_photos).to_csv(os.path.join(OUT_DIR, 'q1_per_photo.csv'), index=False)
summary['q1_concentration_error'] = q1.to_dict('records')
summary['q1_type_accuracy'] = q1_type.to_dict('records')
summary['q1_paired'] = q1_pairs.to_dict('records')
save_summary(summary)
print(q1.to_string(index=False)); print(); print(q1_type.to_string(index=False)); print(); print(q1_pairs.to_string(index=False))

## 6 — Question 2: the hide-the-10% test
All photos of a spice are used. The regressor is trained on pure, 5% and 15% photos only (5 rounds, each time
holding back a fifth of those photos so the 5% and 15% answers are also on photos it has not seen). Every 10% photo
is hidden from training in every round; its answer is the average over the 5 rounds.

The **reference** run is the same thing with nothing hidden (10% photos take part in training like the others).
The gap between the two shows what hiding the 10% level costs.

Columns: `avg_answer_at_5 / 10 / 15` should climb 5 -> 10 -> 15. `order_5_vs_10` and `order_10_vs_15` are the
ordering scores (50 = coin flip). `hidden_low / mid / high` = share of 10% photos answered below 7.5, between 7.5
and 12.5, above 12.5.

In [ ]:
def run_cv(X, g, photo_ids, photo_label, fit, hidden=None, seed=SEED):
    # Returns a per-photo table of out-of-sample answers. Photos at `hidden`% never take part in training.
    photo_pct = np.array([pct_of(l) for l in photo_label])
    pct_by_photo = dict(zip(photo_ids.tolist(), photo_pct.tolist()))
    row_pct = np.array([pct_by_photo[i] for i in g.tolist()])
    known = np.ones(len(photo_ids), bool) if hidden is None else (photo_pct != hidden)
    k_ids, k_lab, h_ids = photo_ids[known], photo_label[known], photo_ids[~known]
    h_rows = np.isin(g, h_ids)
    pred = dict.fromkeys(photo_ids.tolist(), np.nan)
    hidden_runs = []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr_i, te_i in skf.split(k_ids, k_lab):
        tr_rows, te_rows = np.isin(g, k_ids[tr_i]), np.isin(g, k_ids[te_i])
        f = fit(X[tr_rows], row_pct[tr_rows])
        for i, v in zip(k_ids[te_i], photo_median(f(X[te_rows]), g[te_rows], k_ids[te_i])):
            pred[int(i)] = float(v)
        if len(h_ids):
            hidden_runs.append(photo_median(f(X[h_rows]), g[h_rows], h_ids))
    if len(h_ids):
        for i, v in zip(h_ids, np.mean(hidden_runs, axis=0)):
            pred[int(i)] = float(v)
    return pd.DataFrame({'photo_id': photo_ids, 'label': photo_label, 'true_pct': photo_pct,
                         'answer': [pred[int(i)] for i in photo_ids]})


def level_stats(t):
    tp, a = t.true_pct.values, t.answer.values
    ad = tp > 0
    out = {f'avg_answer_at_{int(lv)}': float(a[tp == lv].mean()) for lv in LEVELS}
    out.update({'order_5_vs_10': ordering_score(a, tp, 5.0, 10.0),
                'order_10_vs_15': ordering_score(a, tp, 10.0, 15.0),
                'order_5_vs_15': ordering_score(a, tp, 5.0, 15.0),
                'rank_corr_adulterated': float(spearmanr(tp[ad], a[ad]).correlation),
                'avg_error_at_10': float(np.abs(a[tp == 10.0] - 10.0).mean())})
    h = a[tp == 10.0]
    out.update({'hidden_low': float((h < 7.5).mean() * 100), 'hidden_mid': float(((h >= 7.5) & (h <= 12.5)).mean() * 100),
                'hidden_high': float((h > 12.5).mean() * 100)})
    return out


q2_rows, q2_photos = [], []
for spice in SPICES:
    D = data[spice]
    hand_ids = np.unique(D['g'])
    E = embed_spice(spice, D['items'])
    emb[spice] = E
    ok = ~np.isnan(E).any(axis=1)
    deep_ids = np.array([i for i in hand_ids if ok[i]])
    feature_sets = {'Handcrafted features (new pipeline)': (D['X'], D['g'], hand_ids),
                    'Frozen ResNet50 features': (E[deep_ids], deep_ids, deep_ids)}
    for fs_name, (X, g, ids) in feature_sets.items():
        labels = np.array([D['items'][i][1] for i in ids])
        for reg_name, fit in REGRESSORS.items():
            for run_name, hidden in [('10% hidden', HIDDEN_LEVEL), ('reference (nothing hidden)', None)]:
                t = run_cv(X, g, ids, labels, fit, hidden=hidden)
                q2_rows.append({'spice': spice, 'features': fs_name, 'regressor': reg_name, 'run': run_name,
                                'n_photos': len(t), 'n_at_10': int((t.true_pct == 10).sum()), **level_stats(t)})
                t.insert(0, 'run', run_name); t.insert(0, 'regressor', reg_name)
                t.insert(0, 'features', fs_name); t.insert(0, 'spice', spice)
                q2_photos.append(t)
    print(f'{spice}: done')

q2 = pd.DataFrame(q2_rows).round(2)
q2_photos = pd.concat(q2_photos)
q2.to_csv(os.path.join(OUT_DIR, 'q2_hide_10_test.csv'), index=False)
q2_photos.round(2).to_csv(os.path.join(OUT_DIR, 'q2_per_photo.csv'), index=False)
summary['q2_hide_10_test'] = q2.to_dict('records')
save_summary(summary)
print(q2.to_string(index=False))

### Chart: what the new pipeline's regressor answers at each true level (10% hidden)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, spice in zip(axes, SPICES):
    t = q2_photos[(q2_photos.spice == spice) & (q2_photos.features == 'Handcrafted features (new pipeline)')
                  & (q2_photos.regressor == 'ExtraTrees (new pipeline)') & (q2_photos.run == '10% hidden')]
    groups = [t.answer[t.true_pct == lv].values for lv in LEVELS]
    bp = ax.boxplot(groups, positions=range(4), widths=0.6, patch_artist=True, showfliers=True)
    for patch, lv in zip(bp['boxes'], LEVELS):
        patch.set_facecolor('#f2b134' if lv == HIDDEN_LEVEL else '#7fa8d1')
    for k, lv in enumerate(LEVELS):
        ax.plot([k - 0.4, k + 0.4], [lv, lv], ls='--', c='black', lw=1.2)
    ax.set_xticks(range(4))
    ax.set_xticklabels(['pure', '5%', '10%\n(hidden)', '15%'])
    ax.set_title(spice.replace('_', ' ').title())
    ax.set_xlabel('True adulteration level')
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel("Regressor's answer (%)")
fig.suptitle('Hide-the-10% test: dashed line = correct answer, orange = level never seen in training')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'hide_10_chart.png'), dpi=160)
plt.show()

## 7 — Question 3: would "ResNet50 + a how-much estimate" work on the same test photos?
Regressors trained on train+val photos, scored on the same test photos as Question 1, **adulterated photos only**
and with no pure/adulterated gate, so this isolates the "how much" part.

In [ ]:
q3_rows = []
for spice in SPICES:
    D, E = data[spice], emb[spice]
    hand_ids = np.unique(D['g'])
    ok = ~np.isnan(E).any(axis=1)
    deep_ids = np.array([i for i in hand_ids if ok[i]])
    feature_sets = {'Handcrafted features (new pipeline)': (D['X'], D['g'], hand_ids),
                    'Frozen ResNet50 features': (E[deep_ids], deep_ids, deep_ids)}
    for fs_name, (X, g, ids) in feature_sets.items():
        pct_by_photo = {int(i): pct_of(D['items'][i][1]) for i in ids}
        row_pct = np.array([pct_by_photo[int(i)] for i in g])
        tr_rows, te_rows = g < D['n_tv'], g >= D['n_tv']
        te_ids = ids[ids >= D['n_tv']]
        true_pct = np.array([pct_by_photo[int(i)] for i in te_ids])
        for reg_name, fit in REGRESSORS.items():
            pred = photo_median(fit(X[tr_rows], row_pct[tr_rows])(X[te_rows]), g[te_rows], te_ids)
            m = conc_metrics(true_pct, pred)
            q3_rows.append({'spice': spice, 'features': fs_name, 'regressor': reg_name,
                            **{k: m[k] for k in ['n_adulterated', 'avg_error_adulterated', 'rmse_adulterated',
                                                 'right_band_pct', 'rank_corr_adulterated',
                                                 'avg_answer_at_5', 'avg_answer_at_10', 'avg_answer_at_15']}})
q3 = pd.DataFrame(q3_rows).round(2)
q3.to_csv(os.path.join(OUT_DIR, 'q3_regressor_alone_test_photos.csv'), index=False)
summary['q3_regressor_alone'] = q3.to_dict('records')
summary['finished'] = True
save_summary(summary)
print(q3.to_string(index=False))
print('\nAll results saved to', OUT_DIR)